# Lab 11 — MNIST & Linear Classifiers

*Ten thousand handwritten digits, and a classifier with no hidden layers that still gets most of them right.*

## Objectives

1. Load MNIST and display a montage
2. Build a linear classifier from a single matrix multiply
3. Understand the role of the bias term
4. Measure accuracy honestly on held-out data

---

In [ ]:
# Run this first, every session.
!pip install -q git+https://github.com/williamedwardhahn/mpcr
from mpcr import *
import torch

# Every lab is device-agnostic: it runs on a GPU runtime or a plain CPU one.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on:', device)

# Bias

In [ ]:
X.shape

In [ ]:
bias = GPU_data(np.ones((1,X.shape[1])))

In [ ]:
bias.shape

In [ ]:
X = torch.concatenate([bias,X])

In [ ]:
X.shape

In [ ]:
plt.hist(X[1:,:100].flatten().cpu().numpy())

In [ ]:
N = 20
M = GPU_data(np.random.rand(N,10,785))

m_best = 0 
acc_best = 0

for i in range(1000000):

    y = torch.argmax((M@X), axis=1) 
    score = ((y == Y).sum(1)/len(Y))
    s = torch.argsort(score,descending=True)
    M = M[s]

    M[1:] = M[0]
    M[N//2:] = 0
    M[1:] += GPU_data(np.random.rand(N-1,10,785))

    # M = torch.tanh(M)
    

    acc = score[s][0].item()

    if acc > acc_best:
        
        m_best = M[0]
        acc_best = acc

        print(i,acc)

In [ ]:
bias = GPU_data(np.ones((1,X_test.shape[1])))

In [ ]:
X_test = torch.concatenate([bias,X_test])

In [ ]:
y_test = torch.argmax((M@X_test), axis=1) 
score = ((y_test == Y_test).sum(1)/len(Y_test))
s = torch.argsort(score,descending=True)
acc_test = score[s][0].item()
acc_test

In [ ]:
X.shape

In [ ]:
M.shape

In [ ]:
plt.imshow(M[0,9,1:].reshape(28,28).cpu().numpy())

In [ ]:
X.shape

In [ ]:
plt.imshow(X[1:,0].reshape(28,28).cpu().numpy())

In [ ]:
N = 10
M = GPU_data(np.random.random((N,10,784)))

m_best = 0 
acc_best = 0

for i in range(10000):

    y = torch.argmax((M@X), axis=1) 
    score = ((y == Y).sum(1)/len(Y))
    s = torch.argsort(score,descending=True)
    M = M[s]

    M[1:] = M[0]
    M[N//2] = 0
    M[1:] += 0.00001*GPU_data(np.random.random((N-1,10,784)))
    

    acc = score[s][0].item()

    if acc > acc_best:
        
        m_best = M[0]
        acc_best = acc

        ym = M@X
        ym = torch.argmax(ym, axis=1)
        ym = torch.mode(ym,0)[0]
        acc2 = (((ym == Y)).sum()/len(Y)).item()

        print(i,acc,acc2)

In [ ]:
m_best = 0 
acc_best = 0

for i in range(10000):

    y = torch.argmax((M@X), axis=1) 
    score = ((y == Y).sum(1)/len(Y))
    s = torch.argsort(score,descending=True)
    M = M[s]

    M[1:] = M[0]
    M[N//2] = 0
    M[1:] += 0.00001*GPU_data(np.random.random((N-1,10,784)))
    

    acc = score[s][0].item()

    if acc > acc_best:
        print(i,acc)
        m_best = M[0]
        acc_best = acc

In [ ]:
ym = M@X
ym = torch.argmax(ym, axis=1)
ym = torch.mode(ym,0)[0]
acc = ((ym == Y)).sum()/len(Y)
acc

In [ ]:
%%capture
pip install wandb

In [ ]:
import wandb as wb

In [ ]:
for n in range(100):

    wb.init(project="MNIST Random Walk 10000 d")#,entity="mathdatasci-2023"
    c = wb.config

    c.h = 0.00000001
    m_best = 0 
    acc_best = 0

    for i in range(10000):
        
        m = m_best  + c.h*GPU_data(np.random.standard_normal((10,784)))

        y = m@X

        y = torch.argmax(y, axis=0)

        acc = ((y == Y)).sum()/len(Y)

        if acc > acc_best:
            m_best = m
            acc_best = acc

        wb.log({"acc_best": acc_best})

In [ ]:
X.shape

In [ ]:
N = 10

M = GPU_data(torch.zeros((N,10,784)))

for n in range(N):

    wb.init(project="MNIST Random Walk 10000 h")#,entity="mathdatasci-2023"
    c = wb.config

    c.h = 0.01
    m_best = 0 
    acc_best = 0

    for i in range(10000):
        
        m = m_best  + c.h*GPU_data(np.random.standard_normal((10,784)))

        y = m@X[:,:1000]

        y = torch.argmax(y, axis=0)

        acc = ((y == Y[:1000])).sum()/len(Y[:1000])

        

        if acc > acc_best:
            m_best = m
            acc_best = acc

        wb.log({"acc_best": acc_best})

    M[n] = m_best

In [ ]:
M.shape
X.shape
ym = M@X
ym.shape
ym = torch.argmax(ym, axis=1)
ym.shape
ym = torch.mode(ym,0)[0]
acc = ((ym == Y)).sum()/len(Y)
acc
ym

In [ ]:
N = 100

M = GPU_data(torch.zeros((N,10,784)))

for n in range(N):

    m_best = 0 
    acc_best = 0

    for i in range(1000):
        
        m = m_best  + 0.001*GPU_data(np.random.standard_normal((10,784)))

        y = m@X

        y = torch.argmax(y, axis=0)

        acc = ((y == Y)).sum()/len(Y)


        if acc > acc_best:
            
            m_best = m
            acc_best = acc

    print(acc.item())
    M[n] = m_best

    if n > 2:
        ym = M[:n]@X
        ym = torch.argmax(ym, axis=1)
        ym = torch.mode(ym,0)[0]
        acc = ((ym == Y)).sum()/len(Y)
        print("Swarm: ",acc.item())

In [ ]:
########################################

In [ ]:
N = 100

M = GPU_data(torch.zeros((N,10,784)))

for n in range(N):

    m_best = 0 
    acc_best = 0

    for i in range(10000):
        
        m = m_best  + 0.001*GPU_data(np.random.standard_normal((10,784)))

        y = m@X[:,:1000]

        y = torch.argmax(y, axis=0)

        acc = ((y == Y[:1000])).sum()/len(Y[:1000])


        if acc > acc_best:
            m_best = m
            acc_best = acc

    print(acc_best.item())
    M[n] = m_best

    if n > 2:
        ym = M[:n]@X
        ym = torch.argmax(ym, axis=1)
        ym = torch.mode(ym,0)[0]
        acc = ((ym == Y)).sum()/len(Y)
        print("Swarm: ",acc.item())

In [ ]:
###################

In [ ]:
n = 100

M = GPU_data(np.random.standard_normal((n,10,784)))

y = M@X

y = torch.argmax(y, axis=0)

ym = torch.mode(ym,0)[0]

acc = ((y == Y)).sum()/len(Y)

In [ ]:
acc

In [ ]:
n = 100

M = GPU_data(np.random.standard_normal((n,10,784)))

M_best = 0 
acc_best = 0

for i in range(10000):
    
    M = M_best  + 0.01*GPU_data(np.random.standard_normal((n,10,784)))

    y = M@X

    y = torch.argmax(y, axis=1)

    y = torch.mode(y,0)[0]

    acc = ((y == Y)).sum()/len(Y)


    if acc > acc_best:
        M_best = M
        acc_best = acc
        print(acc_best.item())


In [ ]:
y=M@X

In [ ]:
y.shape

In [ ]:
###

In [ ]:
def train():

    m_best = 0 
    acc_best = 0

    for i in range(1000):
        
        m = m_best  + 0.001*GPU_data(np.random.standard_normal((10,784)))

        y = m@X

        y = torch.argmax(y, axis=0)

        acc = ((y == Y)).sum()/len(Y)


        if acc > acc_best:
            m_best = m
            acc_best = acc

    return acc_best,m_best

In [ ]:
acc_best,m_best = train()
acc_best.item()

In [ ]:
n = 10

M = GPU_data(np.random.standard_normal((n,10,784)))

for i in range(n):
    print(i)
    
    acc_best,m_best = train()

    M[i] = m_best

In [ ]:
y = M@X

y = torch.argmax(y, axis=1)

y = torch.mode(y,0)[0]

acc = ((y == Y)).sum()/len(Y)

In [ ]:
acc

In [ ]:
def Truncated_Normal(size):

    u1 = torch.rand(size)*(1-np.exp(-2)) + np.exp(-2)
    u2 = torch.rand(size)
    z  = torch.sqrt(-2*torch.log(u1)) * torch.cos(2*np.pi*u2)

    return z

In [ ]:
def R(s):
    # return np.random.standard_normal(s)
    return Truncated_Normal(s)

In [ ]:
def a(x):
    return torch.tanh(x)

In [ ]:
def f(x):
    return a(w2@a(w1@a(w0@x)))

In [ ]:
N_in = 784
N_out = 10
N_hidden = 100

In [ ]:
w0 = GPU(R((N_hidden,N_in)))
w1 = GPU(R((N_hidden,N_hidden)))
w2 = GPU(R((N_out,N_hidden)))
y = f(X)
y = torch.argmax(y, axis=0)

acc = ((y == Y)).sum()/len(Y)
acc

In [ ]:
# w0_best = 0
# w1_best = 0
# w2_best = 0

acc_best = 0

s = 0.001

c = 0

for i in range(100000):

    dw0 = s*GPU(R((N_hidden,N_in)))
    dw1 = s*GPU(R((N_hidden,N_hidden)))
    dw2 = s*GPU(R((N_out,N_hidden)))

    w0 = w0_best  + dw0
    w1 = w1_best  + dw1
    w2 = w2_best  + dw2

    y = f(X)

    y = torch.argmax(y, axis=0)

    acc = ((y == Y)).sum()/len(Y)

    c += 1

    if c > 10:
        s = 0.999*s

    if acc > acc_best:
        print(s,acc.item())
        w0_best  = w0
        w1_best  = w1
        w2_best  = w2
        acc_best = acc

        c = 0 
        s = 1.1*s

In [ ]:
N_in = 784
N_out = 10
N_hidden = 100

w0_best = 0
w1_best = 0
w2_best = 0

acc_best = 0

s = 0.001

for i in range(10000000):

    dw0 = s*GPU(R((N_hidden,N_in)))
    dw1 = s*GPU(R((N_hidden,N_hidden)))
    dw2 = s*GPU(R((N_out,N_hidden)))

    w0 = w0_best  + dw0
    w1 = w1_best  + dw1
    w2 = w2_best  + dw2

    y = f(X)

    y = torch.argmax(y, axis=0)

    acc = ((y == Y)).sum()/len(Y)

    if acc > acc_best:
        print(acc.item())
        w0_best  = w0
        w1_best  = w1
        w2_best  = w2
        acc_best = acc


In [ ]:
def exp_decay(epoch):
   initial_lrate = 0.1
   k = 0.1
   lrate = initial_lrate * exp(-k*t)
   return lrate

---

## Homework

1. Train to at least 85% test accuracy and report your confusion matrix.
2. Show three digits your model gets wrong and argue why they are hard.

Submit as a markdown report in your course GitHub repo, with plots committed alongside it.

<details><summary>Where this lab came from</summary>

- `_source/notebooks/MathDataSci/MNIST__19M60mQf.ipynb` cells 72–147 (49 kept)

</details>